# 04 — Fuzzy Matching

We use the **same dataset as 02–03** (500 left, 500 right, 30 true pairs). In 03, Run A (exact full_name only) found 10 pairs—the ones with identical names. We're still missing the 10 true pairs where the name is a variant (Jon vs Jonathan, Katherine vs Catherine, William vs Bill, etc.). **Fuzzy** matching treats a single string column as *similar* instead of equal: matcher returns a **confidence** score (0–1) per pair and you choose a threshold. Here we run exact to see the gap, then fuzzy on full_name to recover those pairs, and use the same measurement loop to tune the threshold with `find_best_threshold`.

**Back:** [03 The Measurement Loop](03_measurement_loop.ipynb) · **Next:** [05 Design Your Algorithm](05_design_algorithm.ipynb)

## 1. Load data (same as 02–03)

Load **raw** entity resolution data, **standardize** it (same as 02–03): we get aligned schema, `email_clean`, and `full_name`. No separate dataset—the 30 true pairs include 10 where the name is a variant on one side (Jonathan/Jon, Katherine/Catherine, etc.); exact full_name misses those, fuzzy on full_name can find them.

In [1]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher, FuzzyMatcher, find_best_threshold

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_entity_resolution, standardize_for_matching

left_raw, right_raw, ground_truth = load_entity_resolution()
left, right = standardize_for_matching(left_raw, right_raw)
# Ground truth keeps left_id/right_id (evaluator requires those names); predicted matches use id/id_right
matcher = Matcher(left=left, right=right, left_id="id", right_id="id")
print(f"Left: {left.shape[0]} rows. Ground truth: {ground_truth.shape[0]} pairs.")
print(left.select(["first_name", "last_name", "full_name"]).head(5))

Left: 500 rows. Ground truth: 30 pairs.
shape: (5, 3)
┌────────────┬───────────┬─────────────┐
│ first_name ┆ last_name ┆ full_name   │
│ ---        ┆ ---       ┆ ---         │
│ str        ┆ str       ┆ str         │
╞════════════╪═══════════╪═════════════╡
│ Alice      ┆ Smith     ┆ Alice Smith │
│ Bob        ┆ Jones     ┆ Bob Jones   │
│ Carol      ┆ White     ┆ Carol White │
│ Dave       ┆ Brown     ┆ Dave Brown  │
│ Eva        ┆ Garcia    ┆ Eva Garcia  │
└────────────┴───────────┴─────────────┘


## 2. A single fuzzy match on full_name

**Fuzzy** matching treats a string column as *similar* instead of equal: `match(on=["field"], matching_algorithm=FuzzyMatcher(threshold=...))` returns pairs above the threshold with a `confidence` score (0–1). Here we run one fuzzy match on `full_name` at threshold 0.8. Next we'll adjust to 0.9, evaluate both, then use `find_best_threshold` to add a third run.

In [2]:
results_fuzzy_v1 = matcher.match(on=["full_name"], matching_algorithm=FuzzyMatcher(threshold=0.8))
print(f"Fuzzy (threshold=0.8): {results_fuzzy_v1.count} matches")
print(results_fuzzy_v1.matches.select(["id", "id_right", "confidence", "full_name", "full_name_right"]).head(5))

Fuzzy (threshold=0.8): 681 matches
shape: (5, 5)
┌──────────┬───────────┬────────────┬────────────────┬─────────────────┐
│ id       ┆ id_right  ┆ confidence ┆ full_name      ┆ full_name_right │
│ ---      ┆ ---       ┆ ---        ┆ ---            ┆ ---             │
│ str      ┆ str       ┆ f64        ┆ str            ┆ str             │
╞══════════╪═══════════╪════════════╪════════════════╪═════════════════╡
│ left_189 ┆ right_30  ┆ 0.809848   ┆ Ashley Hill    ┆ Alex Hill       │
│ left_270 ┆ right_386 ┆ 0.818946   ┆ Steven Flores  ┆ Stephen Brooks  │
│ left_479 ┆ right_77  ┆ 0.837363   ┆ Carol Roberts  ┆ Carolyn Hughes  │
│ left_29  ┆ right_487 ┆ 0.87619    ┆ Deborah Wright ┆ Deborah Edwards │
│ left_399 ┆ right_437 ┆ 0.837363   ┆ Carol Roberts  ┆ Carolyn Hughes  │
└──────────┴───────────┴────────────┴────────────────┴─────────────────┘


## 3. Manual thresholds and the measurement loop

Same loop as 03: evaluate, then compare. Run one—evaluate fuzzy v1 (0.8) against ground truth and check its metrics. Then run another—evaluate fuzzy v2 (0.9) and see how precision, recall, and F1 change as we raise the threshold. Below we also show *which* matches disappear when we go from 0.8 to 0.9: some are true pairs (recall drops) and some are false positives (precision goes up). Next we'll add fuzzy v3 (best threshold) and compare all runs.

In [3]:
# Run one: evaluate fuzzy v1 (0.8 from Section 2)
metrics_fuzzy_v1 = results_fuzzy_v1.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
print(f"Fuzzy v1 (0.8): precision={metrics_fuzzy_v1['precision']:.2%}, recall={metrics_fuzzy_v1['recall']:.2%}, F1={metrics_fuzzy_v1['f1']:.2%}")

# Adjust to 0.9: run fuzzy at 0.9, then evaluate
results_fuzzy_v2 = matcher.match(on=["full_name"], matching_algorithm=FuzzyMatcher(threshold=0.9))
metrics_fuzzy_v2 = results_fuzzy_v2.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
print(f"Fuzzy v2 (0.9): precision={metrics_fuzzy_v2['precision']:.2%}, recall={metrics_fuzzy_v2['recall']:.2%}, F1={metrics_fuzzy_v2['f1']:.2%}")

# Matches that disappear when we raise threshold 0.8 → 0.9 (helps explain precision/recall)
dropped = results_fuzzy_v1.matches.join(
    results_fuzzy_v2.matches.select(["id", "id_right"]), on=["id", "id_right"], how="anti"
)
gt_with_flag = ground_truth.select(pl.col("left_id"), pl.col("right_id"), pl.lit(True).alias("true_pair"))
with_gt = dropped.join(gt_with_flag, left_on=["id", "id_right"], right_on=["left_id", "right_id"], how="left"
).with_columns(pl.col("true_pair").fill_null(False))
sample = with_gt.select(["full_name", "full_name_right", "confidence", "true_pair"]).sort("true_pair", descending=True).head(6)
print("\nSample of pairs dropped at 0.9 (true_pair=True: in ground truth → recall; False: false positive → precision):")
sample

Fuzzy v1 (0.8): precision=3.96%, recall=90.00%, F1=7.59%
Fuzzy v2 (0.9): precision=32.08%, recall=56.67%, F1=40.96%

Sample of pairs dropped at 0.9 (true_pair=True: in ground truth → recall; False: false positive → precision):


full_name,full_name_right,confidence,true_pair
str,str,f64,bool
"""Daniel Lewis""","""Dan Lewis""",0.889815,true
"""Robert Wilson""","""R. Wilson""",0.851282,true
"""Elizabeth Hall""","""Liz Hall""",0.815476,true
"""Nicholas Scott""","""Nick Scott""",0.867407,true
"""Patricia Clark""","""Pat Clark""",0.864815,true
"""Christopher Lee""","""Chris Lee""",0.897778,true


## 4. Tune the threshold with find_best_threshold

Picking a threshold by hand is guesswork. Better: run fuzzy with a *low* threshold so you get many scored pairs, then use `find_best_threshold` to sweep and find the cutoff that maximizes F1 on your ground truth. On this data, exact full_name finds 10; fuzzy on full_name can recover the 10 name-variant pairs (20 from name in total). The same measurement loop as 03—evaluate, compare, decide.

In [ ]:
fuzzy_low = matcher.match(on=["full_name"], matching_algorithm=FuzzyMatcher(threshold=0.5))
best = find_best_threshold(fuzzy_low.matches, ground_truth, left_id_col="id", right_id_col="id_right")

print(f"Best threshold: {best['best_threshold']}")
print(f"  At that threshold: precision={best['best_precision']:.2%}, recall={best['best_recall']:.2%}, F1={best['best_f1']:.2%}")
print("\nSweep (threshold → precision, recall, F1):")
curve = best["curve"]
curve_df = pl.DataFrame({
    "threshold": [c["threshold"] for c in curve],
    "precision": [f"{c['precision']:.2%}" for c in curve],
    "recall": [f"{c['recall']:.2%}" for c in curve],
    "f1": [f"{c['f1']:.2%}" for c in curve],
})
curve_df.tail()

Best threshold: 0.95
  At that threshold: precision=100.00%, recall=43.33%, F1=60.47%

Sweep (threshold → precision, recall, F1):


threshold,precision,recall,f1
f64,str,str,str
0.8,"""3.96%""","""90.00%""","""7.58%"""
0.85,"""7.89%""","""80.00%""","""14.37%"""
0.9,"""41.46%""","""56.67%""","""47.89%"""
0.95,"""100.00%""","""43.33%""","""60.47%"""
1.0,"""100.00%""","""33.33%""","""50.00%"""


## 5. Compare exact and all three fuzzy runs

Run fuzzy again at the **best** threshold from `find_best_threshold` (fuzzy v3), evaluate it, then put **exact** full_name and all three fuzzy runs side by side—same comparison loop as 03. Exact, fuzzy v1 (0.8), fuzzy v2 (0.9), fuzzy v3 (best).

In [ ]:
# Exact full_name (as in 03 Run A), and fuzzy v3 at the best threshold
results_exact = matcher.match(on="full_name")
metrics_exact = results_exact.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
results_fuzzy_v3 = matcher.match(on=["full_name"], matching_algorithm=FuzzyMatcher(threshold=best["best_threshold"]))
metrics_fuzzy_v3 = results_fuzzy_v3.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")

# Compare exact + fuzzy v1, v2, v3 (same loop as 03)
comparison = pl.DataFrame({
    "run": [
        "exact full_name",
        "fuzzy v1 (0.80)",
        "fuzzy v2 (0.90)",
        f"fuzzy v3 (best {best['best_threshold']})",
    ],
    "precision": [
        f"{metrics_exact['precision']:.2%}",
        f"{metrics_fuzzy_v1['precision']:.2%}",
        f"{metrics_fuzzy_v2['precision']:.2%}",
        f"{metrics_fuzzy_v3['precision']:.2%}",
    ],
    "recall": [
        f"{metrics_exact['recall']:.2%}",
        f"{metrics_fuzzy_v1['recall']:.2%}",
        f"{metrics_fuzzy_v2['recall']:.2%}",
        f"{metrics_fuzzy_v3['recall']:.2%}",
    ],
    "f1": [
        f"{metrics_exact['f1']:.2%}",
        f"{metrics_fuzzy_v1['f1']:.2%}",
        f"{metrics_fuzzy_v2['f1']:.2%}",
        f"{metrics_fuzzy_v3['f1']:.2%}",
    ],
})
comparison

run,precision,recall,f1
str,str,str,str
"""exact full_name""","""100.00%""","""33.33%""","""50.00%"""
"""fuzzy v1 (0.80)""","""3.96%""","""90.00%""","""7.59%"""
"""fuzzy v2 (0.90)""","""32.08%""","""56.67%""","""40.96%"""
"""fuzzy v3 (best 0.95)""","""100.00%""","""43.33%""","""60.47%"""


You now have **exact** (02), the **measurement loop** (03), and **fuzzy** (04). Next we put everything together in [05 Design Your Algorithm](05_design_algorithm.ipynb), then optional [06 Blocking](06_blocking.ipynb) and [07 Deduplication](07_deduplication.ipynb).